In [1]:
import pandas as pd
import os

# notebook cwd is /notebooks, so go up one level to reach project root
data_path = os.path.join("..", "data", "raw", "SoftwareDefectDataset.csv")
print("Looking for file at:", os.path.abspath(data_path))

df = pd.read_csv(data_path)

print("\n=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(list(df.columns))

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== HEAD ===")
print(df.head())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== TARGET COLUMN CANDIDATES (search) ===")
for col in df.columns:
    if any(k in col.lower() for k in ["defect", "bug", "label", "target", "class"]):
        print(col, "->", df[col].unique()[:10])

Looking for file at: c:\Users\vedlo\software-defect-risk-engine\data\raw\SoftwareDefectDataset.csv

=== SHAPE ===
(1000, 11)

=== COLUMNS ===
['LOC', 'CYCLO', 'LENGTH', 'VOLUME', 'DIFFICULTY', 'INT_FAN_IN', 'INT_FAN_OUT', 'NUM_OPERATORS', 'NUM_OPERANDS', 'BRANCH_COUNT', 'DEFECT_LABEL']

=== DTYPES ===
LOC              float64
CYCLO            float64
LENGTH           float64
VOLUME           float64
DIFFICULTY       float64
INT_FAN_IN       float64
INT_FAN_OUT      float64
NUM_OPERATORS    float64
NUM_OPERANDS     float64
BRANCH_COUNT     float64
DEFECT_LABEL       int64
dtype: object

=== HEAD ===
        LOC     CYCLO    LENGTH    VOLUME  DIFFICULTY  INT_FAN_IN  \
0  0.779239  0.478261  0.274048  0.544918    0.564121    0.222222   
1  0.595156  0.608696  0.742561  0.758597    0.450649    0.222222   
2  0.895502  0.000000  0.968166  0.754277    0.672996    1.000000   
3  0.782007  0.565217  0.164706  0.017766    0.584106    0.000000   
4  0.757785  0.217391  0.560900  0.126125    0.52

In [2]:
print(df.isnull().sum())
print("\nTotal duplicate rows:", df.duplicated().sum())

LOC              0
CYCLO            0
LENGTH           0
VOLUME           0
DIFFICULTY       0
INT_FAN_IN       0
INT_FAN_OUT      0
NUM_OPERATORS    0
NUM_OPERANDS     0
BRANCH_COUNT     0
DEFECT_LABEL     0
dtype: int64

Total duplicate rows: 0


In [3]:
print(df[["LOC", "CYCLO", "BRANCH_COUNT", "DEFECT_LABEL"]].describe())

               LOC        CYCLO  BRANCH_COUNT  DEFECT_LABEL
count  1000.000000  1000.000000   1000.000000   1000.000000
mean      0.520711     0.493304      0.514643      0.326000
std       0.289402     0.301158      0.314337      0.468982
min       0.000000     0.000000      0.000000      0.000000
25%       0.278720     0.217391      0.214286      0.000000
50%       0.531834     0.478261      0.571429      0.000000
75%       0.767474     0.739130      0.785714      1.000000
max       1.000000     1.000000      1.000000      1.000000


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the cleaned dataset
processed_data_path = "../data/processed/cleaned_dataset.csv"
df = pd.read_csv(processed_data_path)

# 2. Class Imbalance Analysis
print("=== CLASS DISTRIBUTION ===")
class_counts = df['DEFECT_LABEL'].value_counts()
class_percentages = df['DEFECT_LABEL'].value_counts(normalize=True) * 100

for label, count in class_counts.items():
    pct = class_percentages[label]
    status = "Defective" if label == 1 else "Clean"
    print(f"Class {label} ({status}): {count} samples ({pct:.2f}%)")

# Calculate the imbalance ratio (Majority Class / Minority Class)
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"Imbalance Ratio (Clean to Defective): {imbalance_ratio:.2f}:1")

# 3. Correlation Analysis (Spearman Rank Correlation)
print("\n=== FEATURE CORRELATIONS WITH TARGET ===")
# Compute spearman correlation (robust to non-linear monotonic relationships)
correlations = df.corr(method='spearman')['DEFECT_LABEL'].sort_values(ascending=False)
print(correlations)

# 4. Check for Multicollinearity (High correlation between features)
print("\n=== HIGHLY CORRELATED FEATURE PAIRS (Threshold > 0.85) ===")
corr_matrix = df.corr(method='spearman').abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [
    (column, row, upper_tri[column][row])
    for column in upper_tri.columns
    for row in upper_tri.index
    if upper_tri[column][row] > 0.85
]

for col1, col2, val in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True):
    # Exclude correlation with target label itself
    if "DEFECT_LABEL" not in [col1, col2]:
        print(f"{col1} <-> {col2}: {val:.4f}")


=== CLASS DISTRIBUTION ===
Class 0 (Clean): 674 samples (67.40%)
Class 1 (Defective): 326 samples (32.60%)
Imbalance Ratio (Clean to Defective): 2.07:1

=== FEATURE CORRELATIONS WITH TARGET ===
DEFECT_LABEL     1.000000
BRANCH_COUNT     0.062361
NUM_OPERATORS    0.015881
DIFFICULTY       0.002912
LOC             -0.007250
INT_FAN_OUT     -0.007506
INT_FAN_IN      -0.012656
CYCLO           -0.025094
LENGTH          -0.027118
VOLUME          -0.035495
NUM_OPERANDS    -0.068145
Name: DEFECT_LABEL, dtype: float64

=== HIGHLY CORRELATED FEATURE PAIRS (Threshold > 0.85) ===
